# ABC Bank CLU - Deploy and Test

This notebook:
- loads the CLU project name and model label from `.env`
- deploys the trained model to a deployment named `production`
- polls the deployment job until it finishes
- runs a compact test suite against the deployed CLU model
- produces a test results table with pass/fail checks

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import os, time, json, requests

load_dotenv(".env")

LANGUAGE_ENDPOINT = https://REDACTED_ENDPOINT/
LANGUAGE_KEY = REDACTED_AZURE_KEY

PROJECT_NAME = os.getenv("AZURE_CLU_PROJECT_NAME")
MODEL_LABEL = os.getenv("AZURE_CLU_MODEL_LABEL")
DEPLOYMENT_NAME = os.getenv("AZURE_CLU_DEPLOYMENT_NAME", "abc-bank-clu-production")

AUTHORING_API_VERSION = "2023-04-01"
RUNTIME_API_VERSION = "2024-11-01"

if not LANGUAGE_ENDPOINT or not LANGUAGE_KEY:
    raise ValueError("Missing AZURE_LANGUAGE_ENDPOINT / AZURE_LANGUAGE_KEY (or CQA equivalents) in .env")

print("Endpoint:", LANGUAGE_ENDPOINT)
print("Project:", PROJECT_NAME)
print("Model label:", MODEL_LABEL)
print("Deployment:", DEPLOYMENT_NAME)

In [ ]:
# =========================
# HELPERS
# =========================
def upsert_env_file(path: Path, updates: dict):
    existing = {}

    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            existing[k] = v.strip().strip('"')

    existing.update(updates)

    with path.open("w", encoding="utf-8") as f:
        for k, v in existing.items():
            f.write(f'{k}="{v}"\n')


def auth_headers():
    return {
        "Ocp-Apim-Subscription-Key": LANGUAGE_KEY,
        "Content-Type": "application/json",
    }


def poll_operation(operation_url: str, timeout_seconds: int = 900, interval_seconds: int = 5):
    deadline = time.time() + timeout_seconds

    while time.time() < deadline:
        resp = requests.get(
            operation_url,
            headers={"Ocp-Apim-Subscription-Key": LANGUAGE_KEY},
            timeout=60,
        )
        resp.raise_for_status()
        data = resp.json()

        status = str(data.get("status", "")).lower()
        print("Operation status:", status or data)

        if status in ("succeeded", "success"):
            return data
        if status in ("failed", "cancelled", "canceled"):
            raise RuntimeError(f"Operation failed:\\n{json.dumps(data, indent=2)}")

        time.sleep(interval_seconds)

    raise TimeoutError("Timed out waiting for long-running operation to finish.")


def get_deployment():
    url = (
        f"{LANGUAGE_ENDPOINT.rstrip('/')}"
        f"/language/authoring/analyze-conversations/projects/{PROJECT_NAME}/deployments/{DEPLOYMENT_NAME}"
    )
    params = {"api-version": AUTHORING_API_VERSION}

    resp = requests.get(url, headers=auth_headers(), params=params, timeout=60)
    if resp.status_code == 404:
        return None

    resp.raise_for_status()
    return resp.json()


def deploy_model():
    url = (
        f"{LANGUAGE_ENDPOINT.rstrip('/')}"
        f"/language/authoring/analyze-conversations/projects/{PROJECT_NAME}/deployments/{DEPLOYMENT_NAME}"
    )
    params = {"api-version": AUTHORING_API_VERSION}
    body = {"trainedModelLabel": MODEL_LABEL}

    resp = requests.put(url, headers=auth_headers(), params=params, json=body, timeout=60)

    if resp.status_code not in (200, 202):
        raise RuntimeError(
            f"Deployment request failed.\\n"
            f"Status: {resp.status_code}\\n"
            f"Response: {resp.text}"
        )

    operation_location = resp.headers.get("operation-location") or resp.headers.get("Operation-Location")
    if not operation_location:
        try:
            return resp.json()
        except Exception:
            return {"status": "accepted_without_operation_location", "raw": resp.text}

    print("Deployment request accepted.")
    print("Polling:", operation_location)
    return poll_operation(operation_location)


def analyze_utterance(query: str, language: str = "en"):
    url = f"{LANGUAGE_ENDPOINT.rstrip('/')}/language/:analyze-conversations"
    params = {"api-version": RUNTIME_API_VERSION}

    body = {
        "kind": "Conversation",
        "analysisInput": {
            "conversationItem": {
                "participantId": "1",
                "id": "1",
                "modality": "text",
                "language": language,
                "text": query
            },
            "isLoggingEnabled": False
        },
        "parameters": {
            "projectName": PROJECT_NAME,
            "deploymentName": DEPLOYMENT_NAME,
            "verbose": True
        }
    }

    resp = requests.post(url, headers=auth_headers(), params=params, json=body, timeout=60)
    resp.raise_for_status()
    return resp.json()


def extract_prediction(result_json: dict):
    result = result_json.get("result", {})
    prediction = result.get("prediction", {})

    top_intent = prediction.get("topIntent")
    intents = prediction.get("intents", {})
    entities = prediction.get("entities", [])

    return top_intent, intents, entities


def normalize_text(s):
    return str(s).strip().lower()


def check_expected_entity_texts(actual_entities, expected_texts):
    actual_texts = [normalize_text(ent.get("text", "")) for ent in actual_entities]
    expected = [normalize_text(x) for x in expected_texts]
    return all(any(exp in txt or txt in exp for txt in actual_texts) for exp in expected)


def check_expected_entity_categories(actual_entities, expected_categories):
    actual_categories = {normalize_text(ent.get("category", "")) for ent in actual_entities}
    return all(normalize_text(cat) in actual_categories for cat in expected_categories)

In [ ]:
# =========================
# DEPLOY TRAINED MODEL
# =========================
deployment_before = get_deployment()
if deployment_before:
    print("Existing deployment details found:")
    print(json.dumps(deployment_before, indent=2))
else:
    print("No existing deployment found. A new one will be created.")

deploy_result = deploy_model()
print("Deployment completed.")
print(json.dumps(deploy_result, indent=2))

upsert_env_file(Path(".env"), {
    "AZURE_CLU_DEPLOYMENT_NAME": DEPLOYMENT_NAME,
})
print("Saved deployment name to .env")

In [ ]:
# =========================
# TEST CASES
# =========================
test_cases = [
    {
        "utterance": "What’s my current balance in the savings account?",
        "expected_intent": "CheckBalance",
        "expected_entity_categories": ["account_type"],
        "expected_entity_texts": ["savings account"],
    },
    {
        "utterance": "How much do I have in my fixed deposit account?",
        "expected_intent": "CheckBalance",
        "expected_entity_categories": ["account_type"],
        "expected_entity_texts": ["fixed deposit account"],
    },
    {
        "utterance": "Can you tell me my current account balance?",
        "expected_intent": "CheckBalance",
        "expected_entity_categories": [],
        "expected_entity_texts": [],
    },
    {
        "utterance": "Transfer $5000 from my savings to fixed deposit account.",
        "expected_intent": "TransferFunds",
        "expected_entity_categories": ["amount", "account_type"],
        "expected_entity_texts": ["$5000", "savings", "fixed deposit account"],
    },
    {
        "utterance": "Move $2000 to my current account from savings.",
        "expected_intent": "TransferFunds",
        "expected_entity_categories": ["amount", "account_type"],
        "expected_entity_texts": ["$2000", "current account", "savings"],
    },
    {
        "utterance": "Please transfer $500 to my loan account from savings.",
        "expected_intent": "TransferFunds",
        "expected_entity_categories": ["amount", "account_type"],
        "expected_entity_texts": ["$500", "loan account", "savings"],
    },
    {
        "utterance": "Move funds to savings.",
        "expected_intent": "TransferFunds",
        "expected_entity_categories": ["account_type"],
        "expected_entity_texts": ["savings"],
    },
    {
        "utterance": "What’s the status of my loan application?",
        "expected_intent": "LoanStatus",
        "expected_entity_categories": [],
        "expected_entity_texts": [],
    },
    {
        "utterance": "Has my home loan been approved yet?",
        "expected_intent": "LoanStatus",
        "expected_entity_categories": [],
        "expected_entity_texts": [],
    },
    {
        "utterance": "How’s my car loan application going?",
        "expected_intent": "LoanStatus",
        "expected_entity_categories": [],
        "expected_entity_texts": [],
    },
    {
        "utterance": "Show me the balance in my loan account.",
        "expected_intent": "CheckBalance",
        "expected_entity_categories": ["account_type"],
        "expected_entity_texts": ["loan account"],
    },
    {
        "utterance": "Send $1000 from my savings to loan account.",
        "expected_intent": "TransferFunds",
        "expected_entity_categories": ["amount", "account_type"],
        "expected_entity_texts": ["$1000", "savings", "loan account"],
    },
]

df_test_cases = pd.DataFrame(test_cases)
df_test_cases

In [ ]:
# =========================
# RUN TEST CASES
# =========================
results = []

for case in test_cases:
    utterance = case["utterance"]
    response_json = analyze_utterance(utterance)
    top_intent, intents, entities = extract_prediction(response_json)

    top_intent_score = None
    if isinstance(intents, dict) and top_intent in intents:
        top_intent_score = intents[top_intent].get("confidenceScore")
    elif isinstance(intents, list):
        for item in intents:
            if item.get("category") == top_intent:
                top_intent_score = item.get("confidenceScore")
                break

    actual_entity_categories = [ent.get("category") for ent in entities]
    actual_entity_texts = [ent.get("text") for ent in entities]

    intent_pass = top_intent == case["expected_intent"]
    entity_category_pass = check_expected_entity_categories(
        entities,
        case["expected_entity_categories"]
    )
    entity_text_pass = check_expected_entity_texts(
        entities,
        case["expected_entity_texts"]
    ) if case["expected_entity_texts"] else True

    results.append({
        "utterance": utterance,
        "expected_intent": case["expected_intent"],
        "actual_top_intent": top_intent,
        "top_intent_score": top_intent_score,
        "intent_pass": intent_pass,
        "expected_entity_categories": ", ".join(case["expected_entity_categories"]),
        "actual_entity_categories": ", ".join([str(x) for x in actual_entity_categories]),
        "entity_category_pass": entity_category_pass,
        "expected_entity_texts": ", ".join(case["expected_entity_texts"]),
        "actual_entity_texts": ", ".join([str(x) for x in actual_entity_texts]),
        "entity_text_pass": entity_text_pass,
        "overall_pass": intent_pass and entity_category_pass and entity_text_pass,
    })

df_results = pd.DataFrame(results)
df_results

In [ ]:
# =========================
# SUMMARY
# =========================
total = len(df_results)
passed = int(df_results["overall_pass"].sum())
failed = total - passed

print(f"Total tests: {total}")
print(f"Passed: {passed}")
print(f"Failed: {failed}")

df_results[[
    "utterance",
    "expected_intent",
    "actual_top_intent",
    "top_intent_score",
    "overall_pass"
]]

In [ ]:
# =========================
# OPTIONAL: SAVE RESULTS
# =========================
output_dir = Path("analysis_output")
output_dir.mkdir(exist_ok=True)

results_path = output_dir / "abc_bank_clu_test_results.csv"
df_results.to_csv(results_path, index=False, encoding="utf-8-sig")

print(f"Saved test results to: {results_path.resolve()}")